In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('data/processed/cleaned_data.csv')

In [3]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.shape

(7010, 20)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7010 entries, 0 to 7009
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7010 non-null   object 
 1   SeniorCitizen     7010 non-null   int64  
 2   Partner           7010 non-null   object 
 3   Dependents        7010 non-null   object 
 4   tenure            7010 non-null   int64  
 5   PhoneService      7010 non-null   object 
 6   MultipleLines     7010 non-null   object 
 7   InternetService   7010 non-null   object 
 8   OnlineSecurity    7010 non-null   object 
 9   OnlineBackup      7010 non-null   object 
 10  DeviceProtection  7010 non-null   object 
 11  TechSupport       7010 non-null   object 
 12  StreamingTV       7010 non-null   object 
 13  StreamingMovies   7010 non-null   object 
 14  Contract          7010 non-null   object 
 15  PaperlessBilling  7010 non-null   object 
 16  PaymentMethod     7010 non-null   object 


In [6]:
df.isnull().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [7]:
y = df['Churn']
X = df.drop('Churn', axis=1)

In [8]:
y = y.map({"No": 0, "Yes": 1})

In [9]:
y.value_counts()

Churn
0    5153
1    1857
Name: count, dtype: int64

In [10]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()

numerical_cols = X.select_dtypes(exclude='object').columns.tolist()

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)

Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [12]:
y

0       0
1       0
2       1
3       0
4       1
       ..
7005    0
7006    0
7007    0
7008    1
7009    0
Name: Churn, Length: 7010, dtype: int64

In [13]:
for col in categorical_cols:
    print(col, X_train[col].nunique())

gender 2
Partner 2
Dependents 2
PhoneService 2
MultipleLines 3
InternetService 3
OnlineSecurity 3
OnlineBackup 3
DeviceProtection 3
TechSupport 3
StreamingTV 3
StreamingMovies 3
Contract 3
PaperlessBilling 2
PaymentMethod 4


In [14]:
# Note:
# The cleaned training dataset currently has no missing numerical values.
# SimpleImputer is still included as a safety layer to handle unexpected
# missing values in future/production inputs.
#
# TotalCharges needs special business-rule handling during deployment:
# if tenure == 0 and TotalCharges is missing, it should be set to 0
# rather than being replaced blindly by the median.
#
# This special TotalCharges rule will be implemented in the production

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [16]:
from sklearn.preprocessing import OneHotEncoder
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [17]:
from sklearn.compose import ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

In [18]:
X_train_processed = preprocessor.fit_transform(X_train)

In [19]:
X_test_processed = preprocessor.transform(X_test)

In [20]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(5608, 45)
(1402, 45)


In [21]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names)

45
['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'cat__gender_Female' 'cat__gender_Male'
 'cat__Partner_No' 'cat__Partner_Yes' 'cat__Dependents_No'
 'cat__Dependents_Yes' 'cat__PhoneService_No' 'cat__PhoneService_Yes'
 'cat__MultipleLines_No' 'cat__MultipleLines_No phone service'
 'cat__MultipleLines_Yes' 'cat__InternetService_DSL'
 'cat__InternetService_Fiber optic' 'cat__InternetService_No'
 'cat__OnlineSecurity_No' 'cat__OnlineSecurity_No internet service'
 'cat__OnlineSecurity_Yes' 'cat__OnlineBackup_No'
 'cat__OnlineBackup_No internet service' 'cat__OnlineBackup_Yes'
 'cat__DeviceProtection_No' 'cat__DeviceProtection_No internet service'
 'cat__DeviceProtection_Yes' 'cat__TechSupport_No'
 'cat__TechSupport_No internet service' 'cat__TechSupport_Yes'
 'cat__StreamingTV_No' 'cat__StreamingTV_No internet service'
 'cat__StreamingTV_Yes' 'cat__StreamingMovies_No'
 'cat__StreamingMovies_No internet service' 'cat__StreamingMovies_Yes'
 'cat__Contract_Mon

In [22]:
import joblib
import os

os.makedirs("artifacts", exist_ok=True)

joblib.dump(
    preprocessor,
    "artifacts/preprocessor.pkl"
)

['artifacts/preprocessor.pkl']

In [23]:
feature_names = preprocessor.get_feature_names_out()

joblib.dump(
    feature_names,
    "artifacts/feature_names.pkl"
)

['artifacts/feature_names.pkl']

## Baseline Model

In [24]:
#LogisticRegression

from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

In [25]:
lr_model.fit(X_train_processed,y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [26]:
y_pred = lr_model.predict(X_test_processed)

In [27]:
y_prob = lr_model.predict_proba(X_test_processed)[:, 1]

In [28]:
y_prob

array([0.36288057, 0.01377037, 0.08886015, ..., 0.19566228, 0.030019  ,
       0.15145272])

In [29]:
# 3. Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob))

Accuracy : 0.8081312410841655
Precision: 0.67
Recall   : 0.5417789757412399
F1 Score : 0.5991058122205664
ROC-AUC  : 0.8472187000818299


In [30]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[932  99]
 [170 201]]


In [31]:
#Your recall = 54.18%.
#Of the customers who actually churned, the model catches about 54%.
#
#It misses about 46% of the actual churners.

In [32]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier(random_state=42)

In [33]:
dt_model.fit(X_train_processed,y_train)

DecisionTreeClassifier(random_state=42)

In [34]:
y_pred_dt = dt_model.predict(X_test_processed)
y_prob_dt = dt_model.predict_proba(X_test_processed)[:, 1]

In [35]:
print("Accuracy :", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall   :", recall_score(y_test, y_pred_dt))
print("F1 Score :", f1_score(y_test, y_pred_dt))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_dt))

Accuracy : 0.7211126961483595
Precision: 0.47297297297297297
Recall   : 0.4716981132075472
F1 Score : 0.47233468286099867
ROC-AUC  : 0.6421172755103908


In [36]:
cm_dt = confusion_matrix(y_test, y_pred_dt)

print(cm_dt)

[[836 195]
 [196 175]]


In [37]:
#Random Forest 
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)
y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf))

cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix:")
print(cm_rf)

Accuracy : 0.7895863052781741
Precision: 0.6366906474820144
Recall   : 0.477088948787062
F1 Score : 0.5454545454545454
ROC-AUC  : 0.8161403499598694
Confusion Matrix:
[[930 101]
 [194 177]]


In [38]:
# AdaBoost
from sklearn.ensemble import AdaBoostClassifier

ada_model = AdaBoostClassifier(
    random_state=42
)

ada_model.fit(X_train_processed, y_train)

y_pred_ada = ada_model.predict(X_test_processed)
y_prob_ada = ada_model.predict_proba(X_test_processed)[:, 1]

In [39]:
print("Accuracy :", accuracy_score(y_test, y_pred_ada))
print("Precision:", precision_score(y_test, y_pred_ada))
print("Recall   :", recall_score(y_test, y_pred_ada))
print("F1 Score :", f1_score(y_test, y_pred_ada))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_ada))

cm_ada = confusion_matrix(y_test, y_pred_ada)

print("Confusion Matrix:")
print(cm_ada)

Accuracy : 0.7945791726105563
Precision: 0.6456140350877193
Recall   : 0.49595687331536387
F1 Score : 0.5609756097560976
ROC-AUC  : 0.8440579240315711
Confusion Matrix:
[[930 101]
 [187 184]]


In [40]:
# Xgboost

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_processed, y_train)

y_pred_xgb = xgb_model.predict(X_test_processed)
y_prob_xgb = xgb_model.predict_proba(X_test_processed)[:, 1]

In [41]:
print("Accuracy :", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall   :", recall_score(y_test, y_pred_xgb))
print("F1 Score :", f1_score(y_test, y_pred_xgb))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb))

cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("Confusion Matrix:")
print(cm_xgb)

Accuracy : 0.776034236804565
Precision: 0.5899053627760252
Recall   : 0.5040431266846361
F1 Score : 0.5436046511627907
ROC-AUC  : 0.8175089738327482
Confusion Matrix:
[[901 130]
 [184 187]]


| Model               |   Accuracy |  Precision |     Recall |         F1 |   ROC-AUC |
| ------------------- | ---------: | ---------: | ---------: | ---------: | --------: |
| Logistic Regression | **80.81%** | **67.00%** | **54.18%** | **59.91%** | **0.847** |
| Decision Tree       |     72.11% |     47.30% |     47.17% |     47.23% |     0.642 |
| Random Forest       |     78.96% |     63.67% |     47.71% |     54.55% |     0.816 |
| AdaBoost            |     79.46% |     64.56% |     49.60% |     56.10% | **0.844** |
| XGBoost             |     77.60% |     59.00% |     50.40% |     54.36% |     0.818 |


## Hyperparameter Tuning

In [42]:
#lr model tuning
from sklearn.model_selection import GridSearchCV

lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100]
}

grid_lr = GridSearchCV(
    estimator=lr,
    param_grid=param_grid_lr,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_lr.fit(X_train_processed, y_train)

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000, random_state=42),
             n_jobs=-1, param_grid={'C': [0.01, 0.1, 1, 10, 100]},
             scoring='roc_auc')

In [43]:
print("Best Parameters:", grid_lr.best_params_)
print("Best CV ROC-AUC:", grid_lr.best_score_)

Best Parameters: {'C': 1}
Best CV ROC-AUC: 0.8447059415299231


In [44]:
best_lr = grid_lr.best_estimator_

y_pred_lr_tuned = best_lr.predict(X_test_processed)
y_prob_lr_tuned = best_lr.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_lr_tuned))
print("Precision:", precision_score(y_test, y_pred_lr_tuned))
print("Recall   :", recall_score(y_test, y_pred_lr_tuned))
print("F1 Score :", f1_score(y_test, y_pred_lr_tuned))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_lr_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr_tuned))

Accuracy : 0.8081312410841655
Precision: 0.67
Recall   : 0.5417789757412399
F1 Score : 0.5991058122205664
ROC-AUC  : 0.8472187000818299
Confusion Matrix:
[[932  99]
 [170 201]]


In [45]:
# random forest tunning
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_rf.fit(X_train_processed, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=-1, random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [None, 5, 10, 15],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='roc_auc')

In [46]:
print("Best Parameters:", grid_rf.best_params_)
print("Best CV ROC-AUC:", grid_rf.best_score_)

Best Parameters: {'max_depth': 10, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
Best CV ROC-AUC: 0.8462119484694665


In [47]:
best_rf = grid_rf.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_processed)
y_prob_rf_tuned = best_rf.predict_proba(X_test_processed)[:, 1]

In [48]:
print("Accuracy :", accuracy_score(y_test, y_pred_rf_tuned))
print("Precision:", precision_score(y_test, y_pred_rf_tuned))
print("Recall   :", recall_score(y_test, y_pred_rf_tuned))
print("F1 Score :", f1_score(y_test, y_pred_rf_tuned))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_rf_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf_tuned))

Accuracy : 0.796718972895863
Precision: 0.6524822695035462
Recall   : 0.49595687331536387
F1 Score : 0.5635528330781011
ROC-AUC  : 0.8423611964413166
Confusion Matrix:
[[933  98]
 [187 184]]


In [49]:
#adaboosting tunning
from sklearn.model_selection import GridSearchCV

ada = AdaBoostClassifier(
    random_state=42
)

param_grid_ada = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.5, 1.0]
}

grid_ada = GridSearchCV(
    estimator=ada,
    param_grid=param_grid_ada,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_ada.fit(X_train_processed, y_train)

print("Best Parameters:", grid_ada.best_params_)
print("Best CV ROC-AUC:", grid_ada.best_score_)

Best Parameters: {'learning_rate': 0.5, 'n_estimators': 300}
Best CV ROC-AUC: 0.8492171443048523


In [50]:
best_ada = grid_ada.best_estimator_
y_pred_ada_tuned = best_ada.predict(X_test_processed)
y_prob_ada_tuned = best_ada.predict_proba(X_test_processed)[:, 1]

In [51]:
print("Accuracy :", accuracy_score(y_test, y_pred_ada_tuned))
print("Precision:", precision_score(y_test, y_pred_ada_tuned))
print("Recall   :", recall_score(y_test, y_pred_ada_tuned))
print("F1 Score :", f1_score(y_test, y_pred_ada_tuned))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_ada_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_ada_tuned))

Accuracy : 0.7988587731811697
Precision: 0.6583629893238434
Recall   : 0.49865229110512127
F1 Score : 0.5674846625766872
ROC-AUC  : 0.8463572644254524
Confusion Matrix:
[[935  96]
 [186 185]]


In [52]:
# xgboost tunning
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid_xgb,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)

grid_xgb.fit(X_train_processed, y_train)

print("Best Parameters:", grid_xgb.best_params_)
print("Best CV ROC-AUC:", grid_xgb.best_score_)

Best Parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best CV ROC-AUC: 0.8498095105683842


In [53]:
best_xgb = grid_xgb.best_estimator_

y_pred_xgb_tuned = best_xgb.predict(X_test_processed)
y_prob_xgb_tuned = best_xgb.predict_proba(X_test_processed)[:, 1]

print("Accuracy :", accuracy_score(y_test, y_pred_xgb_tuned))
print("Precision:", precision_score(y_test, y_pred_xgb_tuned))
print("Recall   :", recall_score(y_test, y_pred_xgb_tuned))
print("F1 Score :", f1_score(y_test, y_pred_xgb_tuned))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb_tuned))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_tuned))

Accuracy : 0.8038516405135521
Precision: 0.6726618705035972
Recall   : 0.5040431266846361
F1 Score : 0.576271186440678
ROC-AUC  : 0.8450226273918239
Confusion Matrix:
[[940  91]
 [184 187]]


| Model               |   Accuracy |  Precision |     Recall |         F1 |    ROC-AUC |
| ------------------- | ---------: | ---------: | ---------: | ---------: | ---------: |
| Logistic Regression |     80.81% | **67.00%** | **54.18%** | **59.91%** | **0.8472** |
| Decision Tree       |     72.11% |     47.30% |     47.17% |     47.23% |     0.6421 |
| Random Forest       |     79.67% |     65.25% |     49.60% |     56.36% |     0.8424 |
| AdaBoost            |     79.89% |     65.84% |     49.87% |     56.75% |     0.8464 |
| XGBoost             | **80.39%** | **67.27%** |     50.40% |     57.63% |     0.8450 |



For churn, we may care particularly about recall because missing a customer who is actually going to churn (FN) can mean losing an opportunity to intervene.
so for it we need to do changes in threshold to decide churn or not
Lower threshold → more customers classified as churn → usually higher recall, but it will effect precision.

## Threshold analysis

Since the objective of a churn prediction system is to identify as many churners as possible, we performed threshold analysis using cross-validated probabilities on the training data.

In [54]:
#lr analysis
from sklearn.model_selection import cross_val_predict

lr_cv_prob = cross_val_predict(
    best_lr,
    X_train_processed,
    y_train,
    cv=5,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

In [55]:
thresholds = [0.30, 0.35, 0.40, 0.50]

for threshold in thresholds:

    y_pred = (y_prob_lr_tuned >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))


Threshold: 0.3
Accuracy : 0.7631954350927247
Precision: 0.537864077669903
Recall   : 0.7466307277628033
F1 Score : 0.6252821670428894

Threshold: 0.35
Accuracy : 0.7767475035663338
Precision: 0.5635964912280702
Recall   : 0.692722371967655
F1 Score : 0.6215235792019347

Threshold: 0.4
Accuracy : 0.7874465049928673
Precision: 0.5924050632911393
Recall   : 0.6307277628032345
F1 Score : 0.6109660574412533

Threshold: 0.5
Accuracy : 0.8081312410841655
Precision: 0.67
Recall   : 0.5417789757412399
F1 Score : 0.5991058122205664


In [56]:
threshold = 0.35

y_pred_lr_threshold = (
    y_prob_lr_tuned >= threshold
).astype(int)

print("Threshold:", threshold)

print("Accuracy :", accuracy_score(y_test, y_pred_lr_threshold))
print("Precision:", precision_score(y_test, y_pred_lr_threshold))
print("Recall   :", recall_score(y_test, y_pred_lr_threshold))
print("F1 Score :", f1_score(y_test, y_pred_lr_threshold))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_lr_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr_threshold))

Threshold: 0.35
Accuracy : 0.7767475035663338
Precision: 0.5635964912280702
Recall   : 0.692722371967655
F1 Score : 0.6215235792019347
ROC-AUC  : 0.8472187000818299

Confusion Matrix:
[[832 199]
 [114 257]]


In [57]:
#a threshold of 0.35 was selected for the Logistic Regression model for further comparison with the other tuned models.
# as we're catching ~71% of actual churners instead of ~55%.

In [58]:
#xgboost analysis
from sklearn.model_selection import cross_val_predict

xgb_cv_prob = cross_val_predict(
    best_xgb,
    X_train_processed,
    y_train,
    cv=5,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

In [59]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

thresholds = [0.30, 0.35, 0.40, 0.50]

for threshold in thresholds:

    y_pred = (xgb_cv_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")
    print("Precision:", precision_score(y_train, y_pred))
    print("Recall   :", recall_score(y_train, y_pred))
    print("F1 Score :", f1_score(y_train, y_pred))


Threshold: 0.3
Precision: 0.5414519906323185
Recall   : 0.7779273216689099
F1 Score : 0.6384976525821596

Threshold: 0.35
Precision: 0.5698749320282762
Recall   : 0.7052489905787349
F1 Score : 0.630375939849624

Threshold: 0.4
Precision: 0.6151882578174857
Recall   : 0.648721399730821
F1 Score : 0.6315099901735998

Threshold: 0.5
Precision: 0.677797833935018
Recall   : 0.5053835800807537
F1 Score : 0.5790285273708559


In [60]:
threshold = 0.35

y_pred_xgb_threshold = (
    y_prob_xgb_tuned >= threshold
).astype(int)

print("Threshold:", threshold)

print("Accuracy :", accuracy_score(y_test, y_pred_xgb_threshold))
print("Precision:", precision_score(y_test, y_pred_xgb_threshold))
print("Recall   :", recall_score(y_test, y_pred_xgb_threshold))
print("F1 Score :", f1_score(y_test, y_pred_xgb_threshold))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_xgb_tuned))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb_threshold))

Threshold: 0.35
Accuracy : 0.7881597717546363
Precision: 0.5822222222222222
Recall   : 0.706199460916442
F1 Score : 0.6382460414129111
ROC-AUC  : 0.8450226273918239

Confusion Matrix:
[[843 188]
 [109 262]]


At the same threshold, XGBoost achieved slightly higher accuracy, precision, recall, and F1 score, while Logistic Regression had a slightly higher ROC-AUC.

XGBoost model was selected as the final model for the churn prediction system.

The final configuration is:

>Model: XGBoost Classifier
>
>Decision Threshold: 0.35

>Accuracy: 78.82%

>Precision: 58.22%

>Recall: 70.62%

>F1 Score: 63.82%

>ROC-AUC: 84.50%

In [66]:
joblib.dump(best_xgb, "artifacts/xgb_model.pkl")

['artifacts/xgb_model.pkl']

In [67]:
best_xgb.get_params

<bound method XGBModel.get_params of XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)>

In [70]:
import sklearn
import xgboost
import shap

print("scikit-learn:", sklearn.__version__)
print("XGBoost:", xgboost.__version__)
print("SHAP:", shap.__version__)

scikit-learn: 1.6.1
XGBoost: 3.4.1
SHAP: 0.52.0
